# PUF-SENTINEL: run the code of the article

This notebook runs the complete artifact on Google Colab. Choose **Runtime > Run all**.
It takes about 10 minutes and needs no installation on your computer.

1. downloads the repository and installs the pinned packages
2. runs the 94 unit tests
3. runs every experiment with small settings (smoke run)
4. regenerates every table and figure of the article from the stored raw results
5. shows the figures
6. runs the validation: 35 checks that the raw files, tables, numbers and claims are consistent

Repository: https://github.com/amjadabbas4646/PUF-SENTINEL

In [ ]:
# 1. Get the code and install the pinned packages (PyTorch from the CPU index)
!git clone -q https://github.com/amjadabbas4646/PUF-SENTINEL.git
%cd PUF-SENTINEL
# Liberation Serif has the metrics of Times New Roman, the font of the article figures
!apt-get -qq install -y fonts-liberation > /dev/null 2>&1; rm -rf ~/.cache/matplotlib
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r environment/requirements-colab.txt

In [ ]:
# 2. Unit tests
!python -m pytest tests -q

In [ ]:
# 3. Public FPGA data (included in the repository; the script verifies the published checksums)
!python scripts/download_public_data.py

In [ ]:
# 4. Smoke run of every experiment with small settings
!python experiments/run_all.py --config configs/smoke.json --jobs 2
!python experiments/run_hardware.py --smoke --jobs 2
!python experiments/run_extensions.py --config configs/extensions_smoke.json --jobs 2
!python experiments/analyze.py --profile smoke
!python experiments/analyze_extensions.py --profile smoke

In [ ]:
# 5. Every table and figure of the article, regenerated from the stored raw results
!python experiments/analyze.py --profile full
!python experiments/analyze_extensions.py --profile full
!python scripts/make_figures.py --profile full --png figures/png

In [ ]:
# 6. Show the figures
from IPython.display import Image, display, Markdown
import glob
for f in sorted(glob.glob("figures/png/*.png")):
    display(Markdown(f"**{f}**"))
    display(Image(filename=f))

In [ ]:
# 7. Validation: tables regenerate byte for byte, numbers equal the stored results, claims hold
!python scripts/validate_outputs.py

In [ ]:
# 8. One of the generated tables (main attack results, LaTeX source)
print(open("tables/tab_attack_matrix.tex").read())